# RAG 07: OWASP Pipeline Walkthrough

A RAG system has two phases.

The indexing phase converts source material into searchable vectors:

```text
PDF -> Markdown -> chunks -> embeddings -> vector store
```

The answering phase uses that index at question time:

```text
question -> retrieved chunks -> grounded model answer
```

This notebook focuses on document preparation and retrieval inspection. The final chat script adds a LangGraph chat flow with message history after the index exists.


## Step 1: Load Shared Helpers

The parser and chunkers are regular Python functions. The same functions can be used from a notebook, an indexing script, or a larger application.

If a file under `modules/07_rag_owasp_llm/shared/` changes while this notebook is open, restart the kernel before rerunning the notebook. The Python kernel keeps imported modules in memory.

In [ ]:
from pathlib import Path
import logging
import os
import sys

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

logging.getLogger("docling").setLevel(logging.WARNING)
logging.getLogger("RapidOCR").setLevel(logging.WARNING)

CURRENT_FOLDER = Path.cwd()
REPO_ROOT = CURRENT_FOLDER.parents[2] if CURRENT_FOLDER.name == "notebooks" else CURRENT_FOLDER
MODULE_ROOT = REPO_ROOT / "modules" / "07_rag_owasp_llm"
if str(MODULE_ROOT) not in sys.path:
    sys.path.append(str(MODULE_ROOT))

from shared.chunkers import SemanticChunkOutput, gemma_semantic_chunks, recursive_chunks
from shared.pdf_markdown import pdf_to_markdown

load_dotenv(REPO_ROOT / ".env")

PDF_PATH = REPO_ROOT / "data" / "owasp_top10_llm_applications.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Missing PDF at {PDF_PATH}")

print({"pdf_path": str(PDF_PATH)})


## Step 2: Convert PDF To Markdown With Docling

Docling converts the PDF into a structured document and exports Markdown. Markdown gives the chunker useful boundaries such as headings, paragraphs, lists, and tables.

This PDF needs full-page OCR in Docling. Without it, Docling preserves section headings but treats most body regions as images. Full-page OCR can take a few minutes because Docling reads the visual page content instead of only embedded PDF text.


In [ ]:
markdown = pdf_to_markdown(PDF_PATH)
markdown_lines = [line for line in markdown.splitlines() if line.strip()]
headings = [line for line in markdown_lines if line.startswith("#")]

print({"markdown_characters": len(markdown), "headings_found": len(headings)})
print("\nFirst headings:")
print("\n".join(headings[:12]))
print("\nFirst extracted lines:")
print("\n".join(markdown_lines[:20]))


## Step 3: Chunking Strategy A - Recursive Chunking

Recursive splitting tries larger separators first, then falls back to smaller ones.

The `chunk_overlap` setting keeps part of one chunk inside the next chunk.

In [ ]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180

recursive_result = recursive_chunks(
    markdown,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunk_lengths = [len(chunk) for chunk in recursive_result]

print(
    {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "recursive_chunks": len(recursive_result),
        "min_chars": min(chunk_lengths),
        "max_chars": max(chunk_lengths),
        "avg_chars": round(sum(chunk_lengths) / len(chunk_lengths), 1),
    }
)

print("\nChunk 1 ending:")
print(recursive_result[0][-700:])
print("\nChunk 2 beginning:")
print(recursive_result[1][:700])


## Local Ollama Runtime

The next cells call local Ollama models.

Start the Ollama API before running them:

```bash
ollama serve
```

`ollama serve` keeps running in that terminal. Open another terminal and pull the models once:

```bash
ollama pull embeddinggemma
ollama pull gemma4:e4b
```

The notebook connects to Ollama through `OLLAMA_BASE_URL`, which defaults to `http://localhost:11434`.

## Step 4: Chunking Strategy B - Gemma Semantic Chunking

Semantic chunking uses a model to split text around meaning instead of only character count or separators.

The implementation is a LangChain chain:

```text
ChatPromptTemplate -> ChatOllama.with_structured_output(SemanticChunkOutput)
```

`SemanticChunkOutput` is a Pydantic schema with one field: `chunks: list[str]`.

The next retrieval step uses these semantic chunks directly.


In [ ]:
semantic_chunks = gemma_semantic_chunks(
    markdown,
    max_section_chars=2000,
)

print({"semantic_chunks": len(semantic_chunks)})

for index, chunk in enumerate(semantic_chunks[:3], start=1):
    print(f"\nSemantic chunk {index}:")
    print(chunk[:900])


## Step 5: Embedding And Retrieval Check

A vector store owns the similarity-search step. It embeds the documents, stores the vectors, embeds the query, compares vectors, and returns the closest `Document` objects.

`InMemoryVectorStore` is useful here because it shows the retrieval flow without creating a persistent database.

This step indexes the semantic chunks from step 4. Each chunk becomes one LangChain `Document`.


In [ ]:
documents = [
    Document(
        page_content=chunk,
        metadata={
            "source": str(PDF_PATH),
            "chunk_index": index,
        },
    )
    for index, chunk in enumerate(semantic_chunks, start=1)
]

embeddings = OllamaEmbeddings(
    model=os.getenv("OLLAMA_EMBEDDING_MODEL", "embeddinggemma"),
    base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
)

vector_store = InMemoryVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
)

print({"documents_indexed": len(documents)})


In [ ]:
retrieved_documents = vector_store.similarity_search_with_score(
    "What is prompt injection?",
    k=3,
)

for rank, (document, score) in enumerate(retrieved_documents, start=1):
    print()
    print(
        {
            "rank": rank,
            "score": round(float(score), 4),
            "chunk_index": document.metadata["chunk_index"],
        }
    )
    print(document.page_content[:900])


## Step 6: Persistent Index And Chat Runtime

The notebook cells map to the runtime scripts like this:

```text
Docling parsing + semantic chunking -> 19_owasp_llm_build_index.py
embedding + Chroma storage -> 19_owasp_llm_build_index.py
retrieval + graph answer node -> 20_owasp_llm_rag_chat.py
```

The indexing script applies the same parser and semantic chunker to the full document, embeds every chunk, and writes the Chroma index to `data/chroma_owasp_llm`. Re-running it replaces that directory.

```bash
python modules/07_rag_owasp_llm/scripts/19_owasp_llm_build_index.py
```

The chat script loads that existing Chroma index and runs a LangGraph chat flow with one node:

```text
answer
```

The node retrieves documents, builds the system message, calls the model, and returns only the assistant message. The checkpointer stores the `messages` list for the selected `thread_id`; retrieved documents stay local to the node call and can be inspected in LangSmith traces.

```bash
python modules/07_rag_owasp_llm/scripts/20_owasp_llm_rag_chat.py
```

Useful environment variables:

```text
OLLAMA_EMBEDDING_MODEL=embeddinggemma
OLLAMA_CHAT_MODEL=gemma4:e4b
```
